# Phase 0: YOLO11s Training Notebook

This notebook trains YOLO11s for Indian food detection.
- **Task**: detection only (no segmentation)
- **Model**: YOLO11s (small, higher accuracy than YOLOv8n)
- **Data**: IndianFoodNet (30 Indian food classes)
- **Local GPU**: RTX 3050 4GB → batch=4, amp=True, workers=2
- **Kaggle/Colab GPU**: P100/T4 16GB → batch=16, workers=4 (auto-detected)

> **Run in VS Code** (local) or **Kaggle Notebook** (recommended for training).
> Cascade must never execute notebook cells.


## Cell 0 — Environment Detection & Kaggle Dataset Download

In [ ]:
# Cell 0: Environment detection + Kaggle dataset download
import os
from pathlib import Path

KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
CLOUD  = KAGGLE or COLAB

print(f'=== Environment ===')
print(f'Running on Kaggle : {KAGGLE}')
print(f'Running on Colab  : {COLAB}')
print(f'Cloud mode        : {CLOUD}')

if CLOUD:
    # Install dependencies
    os.system('pip install ultralytics roboflow -q')

    # Download IndianFoodNet dataset from Roboflow
    from roboflow import Roboflow
    RF_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'YOUR_ROBOFLOW_API_KEY')
    rf = Roboflow(api_key=RF_API_KEY)
    project = rf.workspace('indianfoodnet').project('indianfoodnet')
    dataset = project.version(1).download('yolov8', location='data/indianfoodnet_yolo')
    print('Dataset downloaded to data/indianfoodnet_yolo')
else:
    print('Local mode — dataset expected at data/indianfoodnet_yolo/')
    print('Add ROBOFLOW_API_KEY to .env if not already downloaded')


## Cell 1 — GPU Check & Imports

In [ ]:
# Cell 1: GPU check + imports
import torch
import yaml
import os
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# CLOUD flag from Cell 0 — re-detect if cells run out of order
KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
CLOUD  = KAGGLE or COLAB

print('=== GPU Check ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU VRAM        : {vram_gb:.1f} GB')
    if vram_gb < 6:
        print('⚠️  <6GB VRAM detected — using local config (batch=4)')
        CLOUD = False  # force local config on low VRAM even if cloud env
else:
    print('WARNING: CUDA not available — training will be extremely slow!')

# Set working directory to project root
project_root = Path.cwd()
if not CLOUD:
    while not (project_root / 'data').exists() and project_root != project_root.parent:
        project_root = project_root.parent
os.chdir(project_root)
print(f'Working directory: {project_root}')

# Config based on environment
BATCH   = 16 if CLOUD else 4     # P100/T4=16, RTX3050 4GB=4
WORKERS = 4  if CLOUD else 2
print(f'\nTraining config  : batch={BATCH}, workers={WORKERS}, cloud={CLOUD}')


## Cell 2 — YOLO11s Detection Training

In [ ]:
# Cell 2: YOLO11s detection training
print('=== Starting YOLO11s Training ===')
print(f'Batch size: {BATCH} | Workers: {WORKERS} | Cloud: {CLOUD}')

# Load pretrained YOLO11s base model
# NOTE: correct model name is 'yolo11s.pt' (no 'v'), not 'yolov11s.pt'
model = YOLO('yolo11s.pt')

training_args = {
    'data'       : 'data/indianfoodnet_yolo/data.yaml',
    'task'       : 'detect',
    'epochs'     : 100,          # increased from 60 for better convergence
    'imgsz'      : 640,
    'batch'      : BATCH,        # 16 on cloud P100/T4, 4 on local 4GB
    'patience'   : 20,           # increased from 15
    'device'     : 0,
    'workers'    : WORKERS,
    'amp'        : True,         # mandatory — cuts VRAM by ~40%
    'save_period': 10,
    'project'    : 'models/runs',
    'name'       : 'yolo11s_indian',
    # Augmentation — critical for food image diversity
    'hsv_h'      : 0.015,        # hue shift handles lighting variation
    'hsv_s'      : 0.7,          # saturation shift
    'hsv_v'      : 0.4,          # brightness shift
    'fliplr'     : 0.5,          # horizontal flip
    'mosaic'     : 1.0,          # 4-image mosaic — major mAP boost
    'mixup'      : 0.1,          # blend two images
    'cls'        : 1.0,          # higher cls loss weight → better confidence scores
}

print(f'Training args: {training_args}')
results = model.train(**training_args)
print('✅ YOLO11s training completed!')


## Cell 3 — Copy best.pt → models/yolo11s_indian.pt

In [ ]:
# Cell 3: Copy best.pt → models/yolo11s_indian.pt
print('=== Saving YOLO11s Model ===')

# Direct path from training args — no directory scan needed
best_model_path = Path('models/runs/yolo11s_indian/weights/best.pt')

# Fallback: scan for latest run if direct path doesn't exist
if not best_model_path.exists():
    runs_dir = Path('models/runs')
    candidates = sorted(
        runs_dir.glob('yolo11s_indian*/weights/best.pt'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if candidates:
        best_model_path = candidates[0]
        print(f'Direct path missed — found via scan: {best_model_path}')
    else:
        print('❌ best.pt not found. Check models/runs/ directory.')
        raise FileNotFoundError('best.pt missing — did training complete?')

dest = Path('models/yolo11s_indian.pt')
dest.parent.mkdir(exist_ok=True)
shutil.copy2(best_model_path, dest)
print(f'✅ Saved: {dest}')
print(f'   Source: {best_model_path}')
print(f'   Size  : {dest.stat().st_size / 1e6:.1f} MB')
print()
print('Next step → update .env:')
print('  YOLO_MODEL_PATH=models/yolo11s_indian.pt')


## Cell 4 — Save Class Names → models/class_names.json

In [ ]:
# Cell 4: Save class names → models/class_names.json
print('=== Extracting Class Names ===')

data_yaml_path = Path('data/indianfoodnet_yolo/data.yaml')
if not data_yaml_path.exists():
    raise FileNotFoundError(f'data.yaml not found at {data_yaml_path}')

with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
print(f'Found {len(class_names)} classes:')
for i, name in enumerate(class_names):
    print(f'  {i:2d}: {name}')

out_path = Path('models/class_names.json')
with open(out_path, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f'\n✅ Saved {len(class_names)} class names to: {out_path}')

# Cross-check expected YOLO classes from project
expected = [
    'biryani','butter_chicken','chapati','chole_bhature','dal_makhani',
    'dal_tadka','dosa','gulab_jamun','idli','jalebi','kadai_paneer',
    'kathi_roll','kheer','kulfi','masala_dosa','medu_vada','naan',
    'pakoda','palak_paneer','paneer_butter_masala','pav_bhaji','poha',
    'puri','rasgulla','ras_malai','samosa','shahi_paneer','uttapam',
    'vada_pav','momos'
]
missing_from_dataset = [c for c in expected if c not in [n.lower().replace(' ','_') for n in class_names]]
if missing_from_dataset:
    print(f'⚠️  Classes in project but not in dataset: {missing_from_dataset}')
else:
    print('✅ All expected food classes present in dataset')


## Cell 5 — Plot Training Loss Curves

In [ ]:
# Cell 5: Plot YOLO11s training loss curves inline
print('=== Plotting Training Curves ===')

# Find results.csv — direct path first, then scan
results_csv = Path('models/runs/yolo11s_indian/results.csv')
if not results_csv.exists():
    candidates = sorted(
        Path('models/runs').glob('yolo11s_indian*/results.csv'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if candidates:
        results_csv = candidates[0]
    else:
        raise FileNotFoundError('results.csv not found — did training complete?')

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()  # strip whitespace from column names
print(f'Loaded {len(df)} epochs from {results_csv}')
print(f'Columns: {df.columns.tolist()}')

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('YOLO11s Training — Indian Food Detection', fontsize=15, fontweight='bold')

# Box loss
axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train', color='#2196F3')
axes[0, 0].plot(df['epoch'], df['val/box_loss'],   label='Val',   color='#FF5722', linestyle='--')
axes[0, 0].set_title('Box Loss'); axes[0, 0].legend(); axes[0, 0].set_xlabel('Epoch')

# Classification loss
axes[0, 1].plot(df['epoch'], df['train/cls_loss'], label='Train', color='#2196F3')
axes[0, 1].plot(df['epoch'], df['val/cls_loss'],   label='Val',   color='#FF5722', linestyle='--')
axes[0, 1].set_title('Classification Loss'); axes[0, 1].legend(); axes[0, 1].set_xlabel('Epoch')

# Precision & Recall
axes[1, 0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#4CAF50')
axes[1, 0].plot(df['epoch'], df['metrics/recall(B)'],    label='Recall',    color='#FF9800')
axes[1, 0].set_title('Precision & Recall'); axes[1, 0].legend(); axes[1, 0].set_xlabel('Epoch')

# mAP
axes[1, 1].plot(df['epoch'], df['metrics/mAP50(B)'],    label='mAP@0.5',     color='#9C27B0')
axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95',color='#607D8B', linestyle='--')
axes[1, 1].set_title('Mean Average Precision'); axes[1, 1].legend(); axes[1, 1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('models/training_curves_yolo11s.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Saved plot to models/training_curves_yolo11s.png')

# Final metrics summary
final = df.iloc[-1]
best_map50 = df['metrics/mAP50(B)'].max()
print(f'\n=== Final YOLO11s Training Metrics ===')
print(f'Total epochs trained : {len(df)}')
print(f'Best mAP@0.5         : {best_map50:.4f}  (epoch {df["metrics/mAP50(B)"].idxmax()})')
print(f'Final mAP@0.5        : {final["metrics/mAP50(B)"]:.4f}')
print(f'Final mAP@0.5:0.95   : {final["metrics/mAP50-95(B)"]:.4f}')
print(f'Final Precision      : {final["metrics/precision(B)"]:.4f}')
print(f'Final Recall         : {final["metrics/recall(B)"]:.4f}')
